# HarmonyRL — training run (Kaggle)

Supervised pretraining on MAESTRO, then PPO fine-tuning.

**Before you run anything, in the right-hand panel:**

1. **Accelerator** → `GPU T4 x2` (or P100)
2. **Internet** → `On`

Run the cells top to bottom. Each stage checks that the previous one produced what
it needs, so a failure stops there instead of cascading.

Kaggle gives ~30 GPU-h/week and a 12h session cap. Stage 1 takes 5–8h on a T4 —
save the checkpoint from the **Output** tab before the session ends.


## 0. Check the GPU


In [ ]:
import sys, torch

assert torch.cuda.is_available(), (
    'No GPU. Set Accelerator to GPU in the right-hand panel and restart.'
)
print('torch     ', torch.__version__)
print('gpu       ', torch.cuda.get_device_name(0))
print('vram (GB) ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('python    ', sys.version.split()[0])


## 1. Get the code


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/SupratikB23/HarmonyRL.git'
BRANCH   = 'main'
REPO     = Path('/kaggle/working/HarmonyRL')

if not REPO.exists():
    !git clone --depth 1 -b $BRANCH $REPO_URL $REPO
else:
    !cd $REPO && git pull

os.chdir(REPO)
!pip install -q -r requirements.txt
print('cwd:', os.getcwd())


## 2. Get MAESTRO

Downloaded straight from Magenta — nothing to upload. ~58 MB, 1276 MIDI files.

Uses `urllib` + `zipfile` so it does not depend on `curl`/`unzip`.

*Alternative:* add MAESTRO via **+ Add Input** and point `data.root` at
`/kaggle/input/...` instead.


In [ ]:
import urllib.request, zipfile
from pathlib import Path

MAESTRO_URL = ('https://storage.googleapis.com/magentadata/datasets/'
               'maestro/v3.0.0/maestro-v3.0.0-midi.zip')
DATA = Path('data/maestro')
DATA.mkdir(parents=True, exist_ok=True)
zip_path = Path('maestro.zip')

if not any(DATA.rglob('*.mid*')):
    print('downloading MAESTRO ...')
    urllib.request.urlretrieve(MAESTRO_URL, zip_path)
    print(f'  {zip_path.stat().st_size / 1e6:.1f} MB, extracting ...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    zip_path.unlink()

n_midi = len(list(DATA.rglob('*.mid*')))
print()
print(f'{n_midi} MIDI files under {DATA}')
if n_midi < 1000:
    raise RuntimeError(
        f'Expected ~1276 MIDI files, found {n_midi}. Download or extract failed; '
        f'delete {DATA} and re-run this cell.'
    )


## 3. Smoke test

Runs the suite before spending GPU hours. If it fails, stop and fix it.


In [ ]:
assert n_midi >= 1000, 'run the MAESTRO cell first'
!python -m pytest tests -q


## 4. Supervised pretraining

~25M params over ~41M tokens. Watch **`val ppl`** — it should fall steadily and land
well under 10. A random model sits near 172 (the vocabulary size).

The first run tokenizes all 1276 files into `.cache/` (a few minutes).

**On a 16 GB card**, if you hit OOM: drop `batch_size` to 8 and `max_seq_len` to 512
in `notebooks/configs/supervised_gpu.yaml`.


In [ ]:
assert n_midi >= 1000, 'run the MAESTRO cell first'
!python -u scripts/train_supervised.py \
    --config notebooks/configs/supervised_gpu.yaml


In [ ]:
from pathlib import Path

sup_ckpt = Path('checkpoints/transformer_supervised.pt')
assert sup_ckpt.exists(), 'pretraining did not produce a checkpoint'
print(f'{sup_ckpt.name}: {sup_ckpt.stat().st_size / 1e6:.1f} MB')


## 5. PPO fine-tuning

Starts from the supervised checkpoint and keeps a frozen copy as the reference policy.

**Watch `R` and `diversity` together:**

| What you see | What it means |
|---|---|
| `R` up, `diversity` ≈ 1.0 | working |
| `R` up, `diversity` falling | **reward hacking** — stop, raise `kl_coef` |
| `kl` drifting up slowly | normal |
| `kl` jumping | policy running from the reference; lower `lr` |


In [ ]:
assert sup_ckpt.exists(), 'run the pretraining cell first'
!python -u scripts/train_rl.py --config notebooks/configs/rl_gpu.yaml


## 6. Generate and inspect

`max_repeat_run` in low single digits is healthy. A large value means the model is
stuck on one note no matter what the reward reports.


In [ ]:
!python -u scripts/infer.py --n_samples 6 --max_new_tokens 1024 \
    --output_dir outputs --no_audio


In [ ]:
import glob
import pretty_midi

midi_files = sorted(glob.glob('outputs/*.mid'))
if not midi_files:
    print('no MIDI written -- check the inference cell above')
for p in midi_files:
    pm = pretty_midi.PrettyMIDI(p)
    notes = pm.instruments[0].notes if pm.instruments else []
    print(f'{p:28s} {len(notes):5d} notes  {pm.get_end_time():6.1f}s')


## 7. Keep your checkpoints

`/kaggle/working` is wiped when the session ends. Download from the **Output** tab,
or save as a Kaggle Dataset so a later session can mount it as an input.


In [ ]:
!ls -lh checkpoints outputs
print()
print('Download checkpoints/*.pt from the Output tab before the session ends.')
